# Q1D | Train Linear Regression | Combined Stage Experiments (impute + transform + scale)

> Takes the discoveries from D-linearRegression-Individual_Stage_Experiments.ipynb (experiments A-E) and combines the more valuable individual findings to a slimmed set of experiments combining impute + transform + scale into a new suite of experiments F+

# Imports

In [1]:
import os
import re
import warnings

import sklearn
print(sklearn.__version__)

import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import BayesianRidge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import ExtraTreesRegressor

from Q1.consts import *
from utils.read_csv_to_df import read_csv_to_df
from Q1.utils.create_pipeline import create_preprocessing, display_pipeline_grid, preprocessing_labels
from Q1.utils.run_experiment import build_pipelines, run_experiment
from Q1.utils.complete_row_folds import Folds, complete_row_folds

from Q1.models import MODELS
from Q1.performance_measures import SCORERS
from Q1.utils.preprocessing.scale import StandardScaler, MinMaxScaler
from Q1.utils.preprocessing.impute import SimpleImputer, KNNImputer, IterativeImputer
from Q1.utils.preprocessing.transform import PowerTransformer, MinShift, on_columns

1.3.2


# Defaults

In [2]:
SEED = 42
np.random.seed(SEED)
pd.set_option("display.float_format", "{:,.4f}".format)

# changing only the fold seed moves mean RMSE by about this much, so smaller gaps are ties
NOISE_FLOOR = 0.002

# no shared CV object here: each experiment builds its folds with complete_row_folds (see Shared Setup),
# which runs KFold(n_splits=4, shuffle=True, random_state=SEED) on the complete rows, the same folds as A to E

# IterativeImputer with a KNN or tree estimator never meets its stopping rule (its fills jump between rounds
# instead of settling), so this warning is expected for those variants and is hidden.
# filterwarnings covers this process (the final refit); PYTHONWARNINGS covers the n_jobs=-1 worker processes,
# which only read it when they start, so restart the kernel if this cell is changed after an experiment has run
IMPUTER_WARNING = "[IterativeImputer] Early stopping criterion not reached"
warnings.filterwarnings("ignore", message=re.escape(IMPUTER_WARNING))
os.environ["PYTHONWARNINGS"] = f"ignore:{IMPUTER_WARNING}"

# Shared Setup

> Every experiment in this notebook (F to I) follows the rules below, so their scores compare directly with each other and with A to E

> **Scoring:** identical to A to E. Every pipeline is scored with `SCORERS` (RMSE, MSE, MAE) through `run_experiment`, which averages the per-fold scores (`cross_validate_means`), then refits on all of the experiment's rows and saves the model

> **Folds (complete-row validation):** `complete_row_folds` runs `KFold(n_splits=4, shuffle=True, random_state=SEED)` on the complete rows only (the 1,374 rows with no missing features), which reproduces the exact folds of A to E. The folds come from the seeded KFold, never hand-picked
> - Pipelines **without an imputer** train on the complete rows of each training fold (the rows passed to `run_experiment` are the complete rows only, since it also refits a final model on them)
> - Pipelines **with an imputer** (G, H) train on those same rows **plus every one of the 185 incomplete rows**. Incomplete rows never appear in a validation fold
> - **Why:** changing the rows changes the folds, so a comparison like "impute (G) vs drop rows (F)" would change both the training and the validation data at once. Fixing validation to the complete rows keeps each experiment to one change
> - The test set has 0 rows with missing values, so validating on complete rows matches the real job

> **Noise floor:** changing only the fold seed moves mean RMSE by about 0.002, so gaps under 0.002 are ties

> **Tie-break rule (decided before any results):** when the top choices tie, prefer the simpler pipeline
> - Yeo-Johnson over Box-Cox (no MinShift needed)
> - StandardScaler over MinMaxScaler (SGD did better with it in Experiment B)
> - no imputer over an imputer

> **Models:** all 7 from `Q1/models.py` at their defaults. Lasso and ElasticNet are expected to collapse as in Experiment B; alpha tuning is deferred to a later notebook

## Plan

| experiment | change                         | question                                                                                                |
|------------|--------------------------------|---------------------------------------------------------------------------------------------------------|
| F          | Transform + Scale              | does a scaler after the power transform let every model reach the transform's gain (about 0.526)?       |
| G          | Add imputation                 | does adding the 185 imputed rows to training beat F's winner on the same validation rows?               |
| H          | Imputer placement (skipped, see G) | does imputing after transform + scale beat imputing raw features?                                 |
| I          | Confirmation across fold seeds | does the winner stay ahead across fold seeds {42, 0, 1, 2, 3}?                                          |

# Fetch Data

> Only the training set is used here. The test set is held back so it plays no part in choosing the model.

In [3]:
train_df = read_csv_to_df(path=TRAIN_PATH)

X = train_df.drop(columns=TARGET)
y = train_df[TARGET]

# Experiment F | Transform + Scale

> **Hypothesis:** a scaler after the power transform lets every model reach the transform's gain (about 0.526 in D), not just OLS and Ridge. In D the unscaled transforms broke SGD, LinearSVR, and svr_linear, so only OLS and Ridge got the gain

> **Compare to:** D (transform without a scaler) and B (scaler without a transform), scored on the same rows and folds

> **Decision:** the transform + scaler of the best pipeline (using the tie-break rule) carries into G

In [4]:
experimentF_model_output_dir = MODEL_OUTPUT_DIR / 'F-TRANSFORM-SCALE-DropRowsWithMissingValues'
experimentF_model_output_dir.mkdir(parents=True, exist_ok=True)

## Cleaning

> transforms and scalers don't fill missing values, so rows with missing values are dropped as in A, B, and D. The folds are built from these same rows (no incomplete rows to add), so they are the exact folds of A to E

> the check cell confirms that, and that the folds G will use (all rows, with the incomplete rows added to training) validate on the same rows as F

In [5]:
def drop_missing_rows(X: pd.DataFrame, y: pd.Series) -> tuple[pd.DataFrame, pd.Series]:
    mask = X.notna().all(axis=1)
    return X[mask], y[mask]

X_experimentF_clean, y_experimentF_clean = drop_missing_rows(X, y)

# no imputer, so train and validate on the complete rows only
experimentF_folds = complete_row_folds(X_experimentF_clean, seed=SEED, add_incomplete_to_train=False)

In [6]:
# check 1: F's folds are the folds of A to E (KFold on the complete rows)
folds_AtoE = list(KFold(n_splits=4, shuffle=True, random_state=SEED).split(X_experimentF_clean))
assert len(experimentF_folds) == len(folds_AtoE)
assert all(
    np.array_equal(train_F, train_AtoE) and np.array_equal(test_F, test_AtoE)
    for (train_F, test_F), (train_AtoE, test_AtoE) in zip(experimentF_folds, folds_AtoE)
)

# check 2: the imputer folds (all rows) validate on F's rows and train on F's rows plus every incomplete row
incomplete_rows = set(X.index[X.isna().any(axis=1)])
for (train_F, test_F), (train_imp, test_imp) in zip(experimentF_folds, complete_row_folds(X, seed=SEED, add_incomplete_to_train=True)):
    assert set(X.index[test_imp]) == set(X_experimentF_clean.index[test_F])
    assert set(X.index[train_imp]) == set(X_experimentF_clean.index[train_F]) | incomplete_rows

print(f"{len(X_experimentF_clean):,} complete rows, {len(incomplete_rows)} incomplete rows, "
      f"validation folds of {[len(test) for _, test in experimentF_folds]} rows")

1,374 complete rows, 185 incomplete rows, validation folds of [344, 344, 343, 343] rows


## PreProcessing

| impute      | transform                                           | scale            |
|-------------|-----------------------------------------------------|------------------|
| passthrough | yeo-johnson, box-cox (each on all 6 skewed columns) | standard, minmax |

> 2 transforms x 2 scalers = 4 preprocessings, x 7 models = 28 pipelines
> - **Transforms:** the two PowerTransformers from Experiment D, unchanged: each applies to **all 6 skewed columns** (|skew| ≥ 0.5, the Q1B rule), and Box-Cox runs after `MinShift(floor=1)` since it needs x > 0 and Generosity has values ≤ 0. `standardize=False` keeps the scaler as the only scaling step
> - **Scalers:** the two from Experiment B, applied to all 8 columns after the transform

In [7]:
# the skewed columns, using the same rule as Q1B: |skew| >= 0.5 on the training set
SKEW_THRESHOLD = 0.5
SKEWED = [c for c in X.columns if abs(X[c].skew()) >= SKEW_THRESHOLD]

# the two PowerTransformers from Experiment D (Box-Cox needs x > 0, so MinShift first for Generosity)
experimentF_transforms = [
	on_columns(PowerTransformer(method="yeo-johnson", standardize=False), SKEWED),
	on_columns(make_pipeline(MinShift(floor=1), PowerTransformer(method="box-cox", standardize=False)), SKEWED),
]

# the two scalers from Experiment B
experimentF_scalers = [StandardScaler(), MinMaxScaler()]

experimentF_preProcessing = [
	create_preprocessing(preprocessing={ "transform": transform, "scale": scaler })
	for transform in experimentF_transforms
	for scaler in experimentF_scalers
]

## Build Pipelines

In [8]:
experimentF_pipelines = build_pipelines(preprocessings=experimentF_preProcessing, models=MODELS, seed=SEED)
# display_pipeline_grid(experimentF_pipelines)

## Cross-Validation Model Fitting

> cross_validate is a function that fits immediately; GridSearchCV is an object that waits for .fit()

> `cv=experimentF_folds` is a list of (train, validation) row positions within `X_experimentF_clean`, so X here must be that same frame

In [9]:
experimentF_df = run_experiment(
    experimentF_pipelines,
    X=X_experimentF_clean,
    y=y_experimentF_clean,
    cv=experimentF_folds,
    scorers=SCORERS,
    output_dir=experimentF_model_output_dir,
    experiment="F",
    cleaning="drop_missing_rows",
)
experimentF_df

,experiment,model,cleaning,impute,transform,scale,rmse,mse,mae,path
0,F,ridge,drop_missing_rows,none,ColumnTransformer(PowerTransformer(method=yeo-...,MinMaxScaler,0.5260,0.2770,0.4080,/Users/mrt/Documents/MrT/IUBloomington/Fall 20...
1,F,ridge,drop_missing_rows,none,ColumnTransformer(PowerTransformer(method=yeo-...,StandardScaler,0.5270,0.2780,0.4090,/Users/mrt/Documents/MrT/IUBloomington/Fall 20...
2,F,sgd,drop_missing_rows,none,ColumnTransformer(PowerTransformer(method=yeo-...,StandardScaler,0.5270,0.2780,0.4090,/Users/mrt/Documents/MrT/IUBloomington/Fall 20...
3,F,linear_regression,drop_missing_rows,none,ColumnTransformer(PowerTransformer(method=yeo-...,StandardScaler,0.5270,0.2780,0.4090,/Users/mrt/Documents/MrT/IUBloomington/Fall 20...
4,F,linear_regression,drop_missing_rows,none,ColumnTransformer(PowerTransformer(method=yeo-...,MinMaxScaler,0.5270,0.2780,0.4090,/Users/mrt/Documents/MrT/IUBloomington/Fall 20...
5,F,ridge,drop_missing_rows,none,ColumnTransformer(MinShift(floor=1)+PowerTrans...,MinMaxScaler,0.5270,0.2780,0.4090,/Users/mrt/Documents/MrT/IUBloomington/Fall 20...
6,F,sgd,drop_missing_rows,none,ColumnTransformer(MinShift(floor=1)+PowerTrans...,StandardScaler,0.5270,0.2780,0.4090,/Users/mrt/Documents/MrT/IUBloomington/Fall 20...
7,F,ridge,drop_missing_rows,none,ColumnTransformer(MinShift(floor=1)+PowerTrans...,StandardScaler,0.5270,0.2780,0.4090,/Users/mrt/Documents/MrT/IUBloomington/Fall 20...
8,F,linear_regression,drop_missing_rows,none,ColumnTransformer(MinShift(floor=1)+PowerTrans...,StandardScaler,0.5270,0.2780,0.4090,/Users/mrt/Documents/MrT/IUBloomington/Fall 20...
9,F,linear_regression,drop_missing_rows,none,ColumnTransformer(MinShift(floor=1)+PowerTrans...,MinMaxScaler,0.5270,0.2780,0.4090,/Users/mrt/Documents/MrT/IUBloomington/Fall 20...


## Experiment F Results

- **Hypothesis mostly holds:** with a scaler after the transform, 5 of 7 models land between 0.526 and 0.532. Ridge with Yeo-Johnson + MinMaxScaler scores 0.526, and ridge, linear_regression, and SGD (StandardScaler) score 0.527 with either transform. That matches D's best (0.526 / 0.527), so scaling adds no gain of its own; it spreads the transform's gain to the other models.
- **Scaling fixed every model that broke in D, with no convergence warnings at all** (D had 40 Liblinear and 20 "Solver terminated early" warnings). svr_linear goes from 13.7 to 608,060 in D to 0.529 / 0.530, LinearSVR from 0.555 to 1.418 to 0.531 / 0.532, and SGD from diverged to 0.527 (StandardScaler).
- **svr_linear and LinearSVR trail on RMSE but lead on MAE:** svr_linear at 0.529 is at the edge of the noise floor and LinearSVR (0.531 / 0.532) is 0.004 to 0.005 behind, but both have the lowest MAE (0.407 to 0.408, against 0.409 for OLS). They minimize an epsilon-insensitive (absolute-style) loss, not squared error, so this is expected.
- **Box-Cox's ridge penalty from D is gone:** ridge with Box-Cox scores 0.527 with either scaler, against 0.531 unscaled in D. Yeo-Johnson and Box-Cox now tie for every model (gaps of 0.001 or less).
- **SGD still prefers StandardScaler:** 0.527 vs 0.545 / 0.546 with MinMaxScaler, the same pattern as B (0.547 vs 0.561).
- **Lasso and ElasticNet still collapse, as expected:** Lasso predicts the mean (1.128, the target's standard deviation) in all 4 preprocessings, ElasticNet does with MinMaxScaler (1.128) and nearly does with StandardScaler (0.858). Alpha tuning is deferred to the next notebook.
- **Carry forward**
  - **Decision:** the top 10 pipelines are within 0.001 of each other, a tie, so the tie-break rule picks **Yeo-Johnson + StandardScaler** for G. StandardScaler is also the only scaler that keeps SGD at the top (0.527 vs 0.545).
  - The bar for G is F's winner on the same validation rows, 0.527 (ridge / linear_regression / SGD with Yeo-Johnson + StandardScaler): an imputer has to reach below 0.525 to count as a gain.
  - Read Lasso and ElasticNet as "untuned" in G and H, not as evidence about imputation.

# Experiment G | Impute + Transform + Scale

> **Hypothesis:** adding the 185 imputed rows to training beats F's winner on the same validation rows

> **Compare to:** F's winner, Yeo-Johnson + StandardScaler at 0.527 (ridge / linear_regression / SGD, chosen by the tie-break rule since F's top 10 were within 0.001). G validates on the same rows and folds as F, so the comparison is direct. Comparing the imputers within G is also clean

> **Decision:** imputation goes forward only if G's best beats F's winner by more than the 0.002 noise floor, ie below 0.525. Otherwise the final pipeline drops incomplete rows, with no imputer, and H is skipped

In [10]:
experimentG_model_output_dir = MODEL_OUTPUT_DIR / 'G-IMPUTE-TRANSFORM-SCALE-KeepAllRows'
experimentG_model_output_dir.mkdir(parents=True, exist_ok=True)

## Cleaning

> no rows are dropped: all 1,559 rows go to `run_experiment`. The imputer is a pipeline step, so it is refit on each training fold only, and validation stays on F's complete rows

> the check cell confirms G validates on exactly F's rows and trains on F's rows plus every incomplete row

In [11]:
X_experimentG_clean, y_experimentG_clean = X, y

# with an imputer, so validate on the complete rows and add every incomplete row to each training fold
experimentG_folds = complete_row_folds(X_experimentG_clean, seed=SEED, add_incomplete_to_train=True)

In [12]:
# G's folds validate on F's rows and train on F's rows plus every incomplete row (the same check as F's check 2)
incomplete_rows = set(X_experimentG_clean.index[X_experimentG_clean.isna().any(axis=1)])
assert len(experimentG_folds) == len(experimentF_folds)
for (train_F, test_F), (train_G, test_G) in zip(experimentF_folds, experimentG_folds):
    assert set(X_experimentG_clean.index[test_G]) == set(X_experimentF_clean.index[test_F])
    assert set(X_experimentG_clean.index[train_G]) == set(X_experimentF_clean.index[train_F]) | incomplete_rows

print(f"{len(X_experimentG_clean):,} rows, {len(incomplete_rows)} incomplete rows in every training fold, "
      f"validation folds of {[len(test) for _, test in experimentG_folds]} rows (F's rows)")

1,559 rows, 185 incomplete rows in every training fold, validation folds of [344, 344, 343, 343] rows (F's rows)


## PreProcessing

| impute                                                   | transform                                | scale    |
|----------------------------------------------------------|------------------------------------------|----------|
| simple (mean, median), knn (5, uniform), iterative (BayesianRidge) | yeo-johnson (on all 6 skewed columns) | standard |

> 4 imputers x 1 transform x 1 scaler = 4 preprocessings, x 7 models = 28 pipelines
> - **Transform + scale:** F's winner, unchanged: Yeo-Johnson on **all 6 skewed columns** (`standardize=False`), then StandardScaler on all 8 columns
> - **Order:** impute, then transform, then scale. The imputers return numpy arrays, but `on_columns` picks its columns by name, so each imputer is set to return a DataFrame (`set_output(transform="pandas")`). That is not a setting of the imputer, so the labels and saved file names are the same as without it

**Imputer selection (4 of C's 11)**

> C tested 11 imputers. Its visible rows were all within the noise floor of each other and of A (best: linear_regression with KNNImputer at 0.547 to 0.548). C's output table showed only its first 5 and last 5 of 77 rows, so "all 11 tie" rests on the visible rows

> G keeps one representative per family, with C's settings:

| imputer                                                 | why                                                                                                                |
|---------------------------------------------------------|--------------------------------------------------------------------------------------------------------------------|
| SimpleImputer(strategy="mean")                          | the simplest baseline                                                                                              |
| SimpleImputer(strategy="median")                        | holds up better than the mean on the 6 skewed columns                                                              |
| KNNImputer(n_neighbors=5, weights="uniform")            | sklearn's default k, the middle of C's 3 / 5 / 10; uniform and distance tied in C's visible rows                    |
| IterativeImputer(estimator=BayesianRidge(), initial_strategy="median") | sklearn's default estimator; the KNeighbors and ExtraTrees variants never met their stopping rule in C and are slower |

> - **To add more:** uncomment a line in `experimentG_imputers`; each adds 7 pipelines
> - **KNNImputer runs on raw features** (impute comes first), so its distances are dominated by Healthy life expectancy, as in C. H tests moving the imputer after transform + scale

In [13]:
# C's 11 imputers with C's exact settings, one per family active (see the table above), so adding one is uncommenting its line
experimentG_imputers = [
	SimpleImputer(strategy="mean"),
	SimpleImputer(strategy="median"),

	# KNNImputer(n_neighbors=3, weights="uniform"),
	# KNNImputer(n_neighbors=3, weights="distance"),
	KNNImputer(n_neighbors=5, weights="uniform"),
	# KNNImputer(n_neighbors=5, weights="distance"),
	# KNNImputer(n_neighbors=10, weights="uniform"),
	# KNNImputer(n_neighbors=10, weights="distance"),

	# BayesianRidge is the default estimator, passed explicitly so the label names it instead of showing None
	IterativeImputer(estimator=BayesianRidge(), initial_strategy="median", random_state=SEED),
	# IterativeImputer(estimator=KNeighborsRegressor(), initial_strategy="median", random_state=SEED),
	# IterativeImputer(estimator=ExtraTreesRegressor(random_state=SEED), initial_strategy="median", random_state=SEED),
]

# F's winner: Yeo-Johnson on the skewed columns (SKEWED from F), then StandardScaler
experimentG_transform = on_columns(PowerTransformer(method="yeo-johnson", standardize=False), SKEWED)
experimentG_scaler = StandardScaler()

experimentG_preProcessing = [
	create_preprocessing(preprocessing={
		# imputers return numpy arrays, but on_columns selects columns by name, so the imputer must return a DataFrame
		"impute": imputer.set_output(transform="pandas"),
		"transform": experimentG_transform,
		"scale": experimentG_scaler,
	})
	for imputer in experimentG_imputers
]

## Build Pipelines

In [14]:
experimentG_pipelines = build_pipelines(preprocessings=experimentG_preProcessing, models=MODELS, seed=SEED)
# display_pipeline_grid(experimentG_pipelines)

## Cross-Validation Model Fitting

> cross_validate is a function that fits immediately; GridSearchCV is an object that waits for .fit()

> `cv=experimentG_folds` is a list of (train, validation) row positions within `X_experimentG_clean` (all 1,559 rows), so X here must be that same frame. The final refit inside `run_experiment` uses all 1,559 rows

In [15]:
experimentG_df = run_experiment(
    experimentG_pipelines,
    X=X_experimentG_clean,
    y=y_experimentG_clean,
    cv=experimentG_folds,
    scorers=SCORERS,
    output_dir=experimentG_model_output_dir,
    experiment="G",
    cleaning="none",
)
experimentG_df

,experiment,model,cleaning,impute,transform,scale,rmse,mse,mae,path
0,G,ridge,none,SimpleImputer(strategy=median),ColumnTransformer(PowerTransformer(method=yeo-...,StandardScaler,0.5270,0.2770,0.4090,/Users/mrt/Documents/MrT/IUBloomington/Fall 20...
1,G,linear_regression,none,SimpleImputer(strategy=median),ColumnTransformer(PowerTransformer(method=yeo-...,StandardScaler,0.5270,0.2770,0.4090,/Users/mrt/Documents/MrT/IUBloomington/Fall 20...
2,G,ridge,none,SimpleImputer(strategy=mean),ColumnTransformer(PowerTransformer(method=yeo-...,StandardScaler,0.5270,0.2780,0.4090,/Users/mrt/Documents/MrT/IUBloomington/Fall 20...
3,G,ridge,none,"IterativeImputer(estimator=BayesianRidge,initi...",ColumnTransformer(PowerTransformer(method=yeo-...,StandardScaler,0.5270,0.2780,0.4100,/Users/mrt/Documents/MrT/IUBloomington/Fall 20...
4,G,linear_regression,none,"IterativeImputer(estimator=BayesianRidge,initi...",ColumnTransformer(PowerTransformer(method=yeo-...,StandardScaler,0.5270,0.2780,0.4100,/Users/mrt/Documents/MrT/IUBloomington/Fall 20...
5,G,linear_regression,none,SimpleImputer(strategy=mean),ColumnTransformer(PowerTransformer(method=yeo-...,StandardScaler,0.5270,0.2780,0.4090,/Users/mrt/Documents/MrT/IUBloomington/Fall 20...
6,G,ridge,none,"KNNImputer(n_neighbors=5,weights=uniform)",ColumnTransformer(PowerTransformer(method=yeo-...,StandardScaler,0.5270,0.2780,0.4100,/Users/mrt/Documents/MrT/IUBloomington/Fall 20...
7,G,linear_regression,none,"KNNImputer(n_neighbors=5,weights=uniform)",ColumnTransformer(PowerTransformer(method=yeo-...,StandardScaler,0.5270,0.2780,0.4100,/Users/mrt/Documents/MrT/IUBloomington/Fall 20...
8,G,sgd,none,SimpleImputer(strategy=median),ColumnTransformer(PowerTransformer(method=yeo-...,StandardScaler,0.5270,0.2780,0.4090,/Users/mrt/Documents/MrT/IUBloomington/Fall 20...
9,G,sgd,none,SimpleImputer(strategy=mean),ColumnTransformer(PowerTransformer(method=yeo-...,StandardScaler,0.5280,0.2790,0.4100,/Users/mrt/Documents/MrT/IUBloomington/Fall 20...


## Experiment G Results

- **Hypothesis rejected:** no imputer beats F's winner. The best G pipelines (ridge and linear_regression with any of the 4 imputers) all score 0.527, the same as F's Yeo-Johnson + StandardScaler on the same validation rows, and none reaches the 0.525 bar.
- **The imputer choice doesn't matter either:** for ridge and linear_regression all 4 imputers score 0.527 (MAE 0.409 to 0.410). SGD (0.527 to 0.528) and svr_linear (0.528 to 0.529) stay within 0.001 of their F scores (0.527, 0.529). LinearSVR with KNN or Iterative (0.529) is 0.002 better than in F (0.531), right at the noise floor, and still behind ridge and linear_regression.
- **Training on 18% more rows (1,215 instead of 1,030 per fold) changed nothing.** With only 8 features, the linear models were likely not short of data, so the 185 extra rows, whose missing values are estimates, add no information the complete rows didn't already give.
- **All 28 pipelines ran without warnings**, including KNNImputer on raw features. The distance problem from C (Healthy life expectancy dominating) did not show up in RMSE.
- **Lasso and ElasticNet are unchanged, as expected:** Lasso still predicts the mean (1.128), and ElasticNet (0.867 to 0.869) is no better than in F (0.858). Both remain untuned.
- **Carry forward**
  - **Decision:** imputation does not go forward. The pipeline is drop incomplete rows → Yeo-Johnson on the 6 skewed columns → StandardScaler, with no imputer. The test set has no missing values, so no imputer is needed at prediction time either.
  - **H is skipped:** its premise (imputation helps) failed, and with all 4 imputers tied there is no imputer placement to improve.
  - I confirms the winner across fold seeds. The closest challengers on the record are ridge with Yeo-Johnson + MinMaxScaler (0.526 in F), Box-Cox + StandardScaler (0.527 in F), and G's median imputer (0.527).

# Experiment H | Imputer Placement (skipped)

> **Skipped:** H only runs if G shows imputation helps (see the Plan)

> G's best tied F's winner (0.527, against the 0.525 bar) and all 4 imputers tied each other, so there is no imputer to place

# Experiment I | Confirmation Across Fold Seeds

> **Hypothesis:** the winning preprocessing (no imputer, Yeo-Johnson on the 6 skewed columns, StandardScaler) stays at least tied with its closest challengers when the folds change, so F's pick was not a lucky split

> **Compare to:** the 3 closest challengers on the record (ridge with Yeo-Johnson + MinMaxScaler at 0.526 in F, Box-Cox + StandardScaler at 0.527 in F, G's median imputer at 0.527), all 7 models, under 5 fold seeds. Within a seed every candidate validates on the same rows, so each fold is a paired comparison

> **Decision (decided before any results):** the winner stays unless a challenger beats it by more than the 0.002 noise floor in mean RMSE over the 20 folds for ridge or linear_regression (the top models). Otherwise the tie-break rule keeps Yeo-Johnson + StandardScaler with no imputer, and that pipeline goes to the tuning notebook. Lasso and ElasticNet are included for completeness but are untuned, so they don't count toward the decision

> **No saved models:** a final model is fit on all of the experiment's rows, so it doesn't depend on the fold seed. I's final models would be identical to the ones F and G already saved, so I saves none and has no output directory

## Cleaning

> the same rows as F and G: the no-imputer candidates use F's complete rows (`X_experimentF_clean`), and the imputer candidate uses all 1,559 rows (`X`), with the incomplete rows added to training only

> for each fold seed, `complete_row_folds` builds both fold sets from the same seeded KFold on the complete rows, so within a seed all 4 candidates validate on the same rows. 5 seeds x 4 folds = 20 validation folds per pipeline. Seed 42 gives F's and G's exact folds

> the check cell confirms both, since the head-to-head pairs the folds by seed and fold number

In [16]:
# the fold seeds; add one here to extend I. Keep SEED (42) in the list: its folds are F's and G's, which the consistency check needs
experimentI_seeds = [42, 0, 1, 2, 3]

# per fold seed: complete-row folds for the no-imputer candidates (as in F), all-row folds for the imputer candidate (as in G)
experimentI_folds = {
	seed: {
		"complete": complete_row_folds(X_experimentF_clean, seed=seed, add_incomplete_to_train=False),
		"all": complete_row_folds(X, seed=seed, add_incomplete_to_train=True),
	}
	for seed in experimentI_seeds
}

In [17]:
# check 1: within each seed, fold k of the imputer candidate validates on the same rows as fold k of the others,
# and trains on the same rows plus every incomplete row, so every fold is a paired comparison
incomplete_rows = set(X.index[X.isna().any(axis=1)])
for seed, folds in experimentI_folds.items():
    assert len(folds["complete"]) == len(folds["all"])
    for (train_c, test_c), (train_a, test_a) in zip(folds["complete"], folds["all"]):
        assert set(X.index[test_a]) == set(X_experimentF_clean.index[test_c])
        assert set(X.index[train_a]) == set(X_experimentF_clean.index[train_c]) | incomplete_rows

# check 2: seed 42 reproduces F's and G's folds exactly
def same_folds(a: Folds, b: Folds) -> bool:
    return len(a) == len(b) and all(
        np.array_equal(train_a, train_b) and np.array_equal(test_a, test_b)
        for (train_a, test_a), (train_b, test_b) in zip(a, b)
    )

assert same_folds(experimentI_folds[SEED]["complete"], experimentF_folds)
assert same_folds(experimentI_folds[SEED]["all"], experimentG_folds)

experimentI_n_folds = sum(len(folds["complete"]) for folds in experimentI_folds.values())
print(f"{len(experimentI_seeds)} seeds x {len(experimentF_folds)} folds = {experimentI_n_folds} validation folds per pipeline, "
      f"paired across the 4 candidates within each seed")

5 seeds x 4 folds = 20 validation folds per pipeline, paired across the 4 candidates within each seed


## PreProcessing

| short name                  | impute          | transform                                       | scale    | rows                                  |
|-----------------------------|-----------------|-------------------------------------------------|----------|---------------------------------------|
| yj+standard (winner)        | passthrough     | yeo-johnson (on all 6 skewed columns)           | standard | complete                              |
| yj+minmax                   | passthrough     | yeo-johnson (on all 6 skewed columns)           | minmax   | complete                              |
| bc+standard                 | passthrough     | box-cox after MinShift(floor=1) (same columns)  | standard | complete                              |
| median+yj+standard          | simple (median) | yeo-johnson (on all 6 skewed columns)           | standard | all (incomplete rows in training only) |

> 1 winner + 3 challengers = 4 preprocessings, x 7 models = 28 pipelines, each scored under 5 fold seeds
> - **Same constructors as F and G:** the PowerTransformers from F (`standardize=False`, on `SKEWED`), the scalers on all 8 columns, and G's median imputer set to return a DataFrame
> - **Models:** the same 7 with `random_state=SEED`; only the fold seed changes
> - **Short names:** the tables below use them instead of the full labels, so they stay readable

In [18]:
# the transforms from F, on the skewed columns (SKEWED from F); Box-Cox needs x > 0, so MinShift first for Generosity
experimentI_yeo_johnson = on_columns(PowerTransformer(method="yeo-johnson", standardize=False), SKEWED)
experimentI_box_cox = on_columns(make_pipeline(MinShift(floor=1), PowerTransformer(method="box-cox", standardize=False)), SKEWED)

# the winner first, then the 3 challengers, keyed by the short names used in the results tables
experimentI_winner = "yj+standard"
experimentI_preProcessing = {
	"yj+standard": create_preprocessing(preprocessing={ "transform": experimentI_yeo_johnson, "scale": StandardScaler() }),
	"yj+minmax": create_preprocessing(preprocessing={ "transform": experimentI_yeo_johnson, "scale": MinMaxScaler() }),
	"bc+standard": create_preprocessing(preprocessing={ "transform": experimentI_box_cox, "scale": StandardScaler() }),
	"median+yj+standard": create_preprocessing(preprocessing={
		# imputers return numpy arrays, but on_columns selects columns by name, so the imputer must return a DataFrame
		"impute": SimpleImputer(strategy="median").set_output(transform="pandas"),
		"transform": experimentI_yeo_johnson,
		"scale": StandardScaler(),
	}),
}

## Build Pipelines

In [19]:
# keyed by (model, short name) instead of the full preprocessing name, so the tables stay readable
experimentI_pipelines = {
	(model_name, short_name): pipeline
	for short_name, preprocessing in experimentI_preProcessing.items()
	for (model_name, _), pipeline in build_pipelines(preprocessings=[preprocessing], models=MODELS, seed=SEED).items()
}
assert len(experimentI_pipelines) == len(experimentI_preProcessing) * len(MODELS)
# display_pipeline_grid(experimentI_pipelines)

## Cross-Validation Model Fitting

> **Same methodology, finer reporting:** every pipeline is scored with the same `SCORERS`, the same fold construction, and the same `cross_validate_folds` call that `cross_validate_means` makes in F and G (both in Q1/utils/run_experiment.py). The only difference: I keeps each fold's score instead of averaging and rounding to 3 decimals, since the gaps being judged are about 0.001 and the head-to-head compares fold by fold

> each pipeline gets the rows and folds of its candidate: `X_experimentF_clean` with the complete-row folds without an imputer, `X` with the all-row folds with one

In [20]:
# cross_validate_folds moved to Q1/utils/run_experiment.py, where cross_validate_means averages it for F and G
from Q1.utils.run_experiment import cross_validate_folds

In [21]:
# one row per (model, preprocessing, seed, fold)
experimentI_fold_rows = []
for seed, folds in experimentI_folds.items():
    for (model_name, short_name), pipeline in experimentI_pipelines.items():
        # with an imputer: all rows and G's kind of folds; without: the complete rows and F's kind of folds
        has_imputer = preprocessing_labels(pipeline.named_steps["preprocess"])["impute"] != "none"
        X_I, y_I = (X, y) if has_imputer else (X_experimentF_clean, y_experimentF_clean)
        folds_I = folds["all"] if has_imputer else folds["complete"]

        scores = cross_validate_folds(pipeline, X=X_I, y=y_I, cv=folds_I, scorers=SCORERS)
        for fold in range(len(folds_I)):
            experimentI_fold_rows.append({
                "model": model_name,
                "preprocessing": short_name,
                "seed": seed,
                "fold": fold,
                **{metric: fold_scores[fold] for metric, fold_scores in scores.items()},
            })

experimentI_folds_df = pd.DataFrame(experimentI_fold_rows)

# one row per (model, preprocessing, seed): the mean of that seed's folds
experimentI_df = (
    experimentI_folds_df
    .groupby(["model", "preprocessing", "seed"], sort=False)[list(SCORERS)]
    .mean()
    .reset_index()
)

## Consistency Check

> seed 42 is F's and G's folds, so each candidate's mean RMSE there, rounded to 3 decimals as `sort_results` does, must equal what F (no imputer) or G (median imputer) reported for the same model and preprocessing. That shows I is on the same scale as F and G

In [22]:
# match on the model and the same impute / transform / scale labels that F's and G's tables use
experimentI_labels = pd.DataFrame([
    {"preprocessing": short_name, **preprocessing_labels(preprocessing)}
    for short_name, preprocessing in experimentI_preProcessing.items()
])
experimentI_check = (
    experimentI_df[experimentI_df["seed"] == SEED]
    .merge(experimentI_labels, on="preprocessing")
    .merge(
        pd.concat([experimentF_df, experimentG_df])[["experiment", "model", "impute", "transform", "scale", "rmse"]],
        on=["model", "impute", "transform", "scale"],
        suffixes=("", "_FG"),
        validate="one_to_one",
    )
)
experimentI_mismatch = experimentI_check[~np.isclose(experimentI_check["rmse"].round(3), experimentI_check["rmse_FG"])]
assert len(experimentI_check) == len(experimentI_pipelines), "a candidate has no match in F or G"
assert experimentI_mismatch.empty, experimentI_mismatch[["model", "preprocessing", "experiment", "rmse", "rmse_FG"]]

print(f"seed {SEED}: all {len(experimentI_check)} pipelines match F's and G's rmse to 3 decimals "
      f"({(experimentI_check['experiment'] == 'F').sum()} from F, {(experimentI_check['experiment'] == 'G').sum()} from G)")

seed 42: all 28 pipelines match F's and G's rmse to 3 decimals (21 from F, 7 from G)


## Results Tables

> **Mean RMSE per seed:** one row per pipeline, with the mean RMSE of each seed's folds, "mean" over all 20 folds, and "std" of the 5 seed means (how much the fold seed alone moves the score), best first. The full per-seed table, with mse and mae too, is `experimentI_df`, and the per-fold one is `experimentI_folds_df`

In [23]:
experimentI_rmse = experimentI_df.pivot(index=["model", "preprocessing"], columns="seed", values="rmse")[experimentI_seeds]
experimentI_rmse["mean"] = experimentI_folds_df.groupby(["model", "preprocessing"])["rmse"].mean()
experimentI_rmse["std"] = experimentI_rmse[experimentI_seeds].std(axis=1)
experimentI_rmse.sort_values("mean")

seed                                     42      0      1      2      3  \
model             preprocessing                                           
ridge             yj+minmax          0.5264 0.5253 0.5266 0.5246 0.5254   
                  median+yj+standard 0.5265 0.5254 0.5267 0.5246 0.5254   
linear_regression median+yj+standard 0.5266 0.5254 0.5267 0.5246 0.5254   
ridge             yj+standard        0.5267 0.5253 0.5269 0.5248 0.5254   
linear_regression yj+minmax          0.5267 0.5253 0.5269 0.5248 0.5254   
                  yj+standard        0.5267 0.5253 0.5269 0.5248 0.5254   
sgd               yj+standard        0.5267 0.5256 0.5264 0.5266 0.5254   
ridge             bc+standard        0.5271 0.5258 0.5274 0.5252 0.5258   
linear_regression bc+standard        0.5271 0.5258 0.5274 0.5252 0.5258   
sgd               median+yj+standard 0.5272 0.5259 0.5270 0.5264 0.5249   
                  bc+standard        0.5271 0.5261 0.5269 0.5270 0.5258   
svr_linear        median+yj+standard 0.5290 0.5269 0.5296 0.5278 0.5275   
                  yj+minmax          0.5287 0.5268 0.5299 0.5277 0.5280   
                  yj+standard        0.5289 0.5266 0.5299 0.5279 0.5280   
                  bc+standard        0.5299 0.5271 0.5306 0.5282 0.5286   
linear_svr        median+yj+standard 0.5304 0.5281 0.5310 0.5283 0.5287   
                  yj+minmax          0.5308 0.5280 0.5310 0.5280 0.5288   
                  yj+standard        0.5306 0.5275 0.5314 0.5283 0.5291   
                  bc+standard        0.5316 0.5282 0.5318 0.5288 0.5294   
sgd               yj+minmax          0.5451 0.5476 0.5467 0.5473 0.5466   
elastic_net       yj+standard        0.8578 0.8573 0.8583 0.8579 0.8598   
                  bc+standard        0.8585 0.8580 0.8590 0.8586 0.8605   
                  median+yj+standard 0.8676 0.8671 0.8680 0.8675 0.8692   
lasso             yj+standard        1.1281 1.1279 1.1283 1.1293 1.1300   
                  yj+minmax          1.1281 1.1279 1.1283 1.1293 1.1300   
                  bc+standard        1.1281 1.1279 1.1283 1.1293 1.1300   
elastic_net       yj+minmax          1.1281 1.1279 1.1283 1.1293 1.1300   
lasso             median+yj+standard 1.1282 1.1281 1.1285 1.1291 1.1298   

seed                                   mean    std  
model             preprocessing                     
ridge             yj+minmax          0.5257 0.0008  
                  median+yj+standard 0.5257 0.0009  
linear_regression median+yj+standard 0.5257 0.0009  
ridge             yj+standard        0.5258 0.0009  
linear_regression yj+minmax          0.5258 0.0009  
                  yj+standard        0.5258 0.0009  
sgd               yj+standard        0.5261 0.0006  
ridge             bc+standard        0.5263 0.0009  
linear_regression bc+standard        0.5263 0.0009  
sgd               median+yj+standard 0.5263 0.0009  
                  bc+standard        0.5266 0.0006  
svr_linear        median+yj+standard 0.5282 0.0011  
                  yj+minmax          0.5282 0.0012  
                  yj+standard        0.5283 0.0012  
                  bc+standard        0.5289 0.0014  
linear_svr        median+yj+standard 0.5293 0.0013  
                  yj+minmax          0.5293 0.0015  
                  yj+standard        0.5293 0.0016  
                  bc+standard        0.5300 0.0016  
sgd               yj+minmax          0.5466 0.0010  
elastic_net       yj+standard        0.8582 0.0010  
                  bc+standard        0.8589 0.0009  
                  median+yj+standard 0.8679 0.0008  
lasso             yj+standard        1.1287 0.0009  
                  yj+minmax          1.1287 0.0009  
                  bc+standard        1.1287 0.0009  
elastic_net       yj+minmax          1.1287 0.0009  
lasso             median+yj+standard 1.1287 0.0007

> **Head-to-head vs the winner:** per model and challenger, over the 20 paired folds (same seed and fold number, so the same validation rows). "gap" is the challenger's mean RMSE minus the winner's (negative means the challenger is better), and "wins" counts the folds where the challenger's RMSE is lower by more than `WIN_TOLERANCE` (1e-9). Smaller gaps are floating-point rounding (ex linear_regression gives the same predictions with either scaler), so they count as ties

In [24]:
experimentI_winner_folds = experimentI_folds_df[experimentI_folds_df["preprocessing"] == experimentI_winner]
experimentI_paired = (
    experimentI_folds_df[experimentI_folds_df["preprocessing"] != experimentI_winner]
    .rename(columns={"preprocessing": "challenger"})
    .merge(
        experimentI_winner_folds[["model", "seed", "fold", "rmse"]],
        on=["model", "seed", "fold"],
        suffixes=("", "_winner"),
        validate="many_to_one",
    )
)
experimentI_paired["gap"] = experimentI_paired["rmse"] - experimentI_paired["rmse_winner"]
# gaps this small are floating-point rounding, not a real difference, so they count as ties instead of wins
WIN_TOLERANCE = 1e-9
# every challenger is paired with the winner on all 20 folds
assert (experimentI_paired.groupby(["model", "challenger"]).size() == experimentI_n_folds).all()

experimentI_head_to_head = (
    experimentI_paired
    .groupby(["model", "challenger"])
    .agg(
        winner_rmse=("rmse_winner", "mean"),
        challenger_rmse=("rmse", "mean"),
        gap=("gap", "mean"),
        wins=("gap", lambda gap: int((gap < -WIN_TOLERANCE).sum())),
    )
    .rename(columns={"wins": f"wins (of {experimentI_n_folds})"})
    # MODELS order, then the challengers in the order of the PreProcessing table
    .reindex(pd.MultiIndex.from_product(
        [list(MODELS), [name for name in experimentI_preProcessing if name != experimentI_winner]],
        names=["model", "challenger"],
    ))
)
experimentI_head_to_head

winner_rmse  challenger_rmse     gap  \
model             challenger                                                 
linear_regression yj+minmax                0.5258           0.5258 -0.0000   
                  bc+standard              0.5258           0.5263  0.0004   
                  median+yj+standard       0.5258           0.5257 -0.0001   
sgd               yj+minmax                0.5261           0.5466  0.0205   
                  bc+standard              0.5261           0.5266  0.0004   
                  median+yj+standard       0.5261           0.5263  0.0002   
ridge             yj+minmax                0.5258           0.5257 -0.0002   
                  bc+standard              0.5258           0.5263  0.0004   
                  median+yj+standard       0.5258           0.5257 -0.0001   
lasso             yj+minmax                1.1287           1.1287  0.0000   
                  bc+standard              1.1287           1.1287  0.0000   
                  median+yj+standard       1.1287           1.1287  0.0000   
elastic_net       yj+minmax                0.8582           1.1287  0.2705   
                  bc+standard              0.8582           0.8589  0.0007   
                  median+yj+standard       0.8582           0.8679  0.0097   
linear_svr        yj+minmax                0.5293           0.5293 -0.0000   
                  bc+standard              0.5293           0.5300  0.0006   
                  median+yj+standard       0.5293           0.5293 -0.0001   
svr_linear        yj+minmax                0.5283           0.5282 -0.0001   
                  bc+standard              0.5283           0.5289  0.0006   
                  median+yj+standard       0.5283           0.5282 -0.0001   

                                      wins (of 20)  
model             challenger                        
linear_regression yj+minmax                      0  
                  bc+standard                    2  
                  median+yj+standard            10  
sgd               yj+minmax                      0  
                  bc+standard                    2  
                  median+yj+standard            11  
ridge             yj+minmax                     13  
                  bc+standard                    2  
                  median+yj+standard            10  
lasso             yj+minmax                      0  
                  bc+standard                    0  
                  median+yj+standard            11  
elastic_net       yj+minmax                      0  
                  bc+standard                    0  
                  median+yj+standard             0  
linear_svr        yj+minmax                      9  
                  bc+standard                    6  
                  median+yj+standard             9  
svr_linear        yj+minmax                     12  
                  bc+standard                    4  
                  median+yj+standard            11

## Experiment I Results

- **Hypothesis holds:** over 20 folds (5 seeds x 4), no challenger beats Yeo-Johnson + StandardScaler by more than 0.0002 for ridge or linear_regression, 10 times smaller than the 0.002 bar. The winner averages 0.5258 for both, against 0.5257 for the best challengers (ridge with MinMaxScaler, and the median imputer).
- **The fold seed moves scores far more than the preprocessing choice:** the winner's seed means range from 0.5248 to 0.5269 for ridge and linear_regression (std 0.0009), while their top 3 candidates stay within 0.0003 of each other on every seed (0.0007 including Box-Cox). This confirms the 0.002 noise floor and shows F's pick was not a lucky split.
- **The challengers are coin flips or worse:** the median imputer wins 10 of 20 folds against the winner for ridge and linear_regression. MinMaxScaler wins 13 of 20 for ridge (gap -0.0002) and 0 for linear_regression, whose predictions don't depend on the scaler. Box-Cox loses consistently: 0.0004 worse for ridge, linear_regression, and SGD, winning only 2 of 20 folds.
- **SGD needs StandardScaler:** with MinMaxScaler it is 0.0205 worse and wins 0 of 20 folds, confirming the B and F pattern.
- **The model ranking is stable across seeds:** ridge and linear_regression 0.5258, SGD 0.5261, svr_linear 0.5283, LinearSVR 0.5293. Lasso (1.1287) still predicts the mean and ElasticNet (0.8582) is far behind. Both are untuned and don't count toward the decision.
- **Carry forward**
  - **Decision:** the winner stays. The final preprocessing for the tuning notebook is: drop incomplete rows → Yeo-Johnson on the 6 skewed columns → StandardScaler, no imputer.
  - **Tuning targets:** alpha for Lasso and ElasticNet (they cannot be judged until then), and alpha for ridge. ridge's small MinMaxScaler edge (13 of 20 folds) suggests its penalty strength matters a little. SGD, svr_linear, and LinearSVR settings are secondary, since they already sit within 0.0035 of OLS.
  - **Bar for tuning:** ridge / linear_regression at 0.5258 over these 20 folds. A tuned model has to beat that by more than 0.002, and tuning comparisons should reuse I's 5-seed procedure so a lucky split can't pick the winner.